# 07 · SHAP evaluation of the frozen LightGBM model
This notebook explains the team's **frozen final model**: the 06b LightGBM pipeline (original class balance), run `20261011_232303_full`, with review threshold **0.0861**. It covers the whole pipeline: the median imputer followed by LightGBM.

**Protocol**
- **Nothing is refitted or retuned.** The pipeline, feature order and threshold are loaded from the run folder and checked against `artifacts/shared/final_model_selection.json`. The held-out scores are recomputed and must match the saved `test_predictions.csv` exactly before any explanation is produced.
- **Data explained: the 1,082 held-out test providers.** The held-out evaluation is already complete (see `reports/final_model_evaluation.md`). Here, test labels are used only after the fact, to group explanations into true/false positives and negatives. Nothing in this notebook feeds back into the model or the threshold.
- **Primary explainer: TreeSHAP, path-dependent, on the log-odds scale.** This is LightGBM's `pred_contrib`, the same values the dashboard shows under "Why it scored here". For every provider, the contributions plus the base value add up exactly to the model's log-odds score. Section 8 cross-checks it against the `shap` package and an interventional (background-data) variant.
- **Scale.** SHAP values are in log-odds. The review threshold 0.0861 corresponds to log-odds −2.36. A contribution of +1 multiplies the odds by about 2.7. Contributions are additive in log-odds, not in probability.
- **Missing values.** For `TopDiagnosisCodeShare`, `TopProcedureCodeShare` and `BenfordMAD`, values are set back to missing where the `…Missing` flag is 1, then median-imputed, as in training. The model therefore sees the training median for those providers. Its view of "missing" comes from the flag and the imputed value together; section 6 analyses them jointly.

**What SHAP does and does not show.** SHAP attributes the *model's* score to its inputs. It describes how the model uses the features, not what causes fraud. `PotentialFraud` is a proxy label, and correlated features can share or trade credit.

All tables and figures are written to `reports/shap_evaluation/`. The run takes a few minutes, mostly in the interventional SHAP and interaction cells of section 8.

In [ ]:
import hashlib
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import lightgbm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn
from IPython.display import display
from scipy.stats import spearmanr

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/final_evaluation.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository containing src/final_evaluation.py.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.final_evaluation import load_saved_results

SEED = 4012                 # project seed
N_BOOT = 1000               # label-stratified provider bootstrap for importance intervals
BACKGROUND_SIZE = 200       # development providers used as background for interventional SHAP
TOP_K = 10                  # features shown in tables and some plots
N_DEPENDENCE = 6            # dependence plots for the top-N features
NEAR_BAND_LOGODDS = 0.5     # "near threshold": |log-odds score - log-odds threshold| <= this
RUN_INTERACTIONS = True     # SHAP interaction values (slowest optional step)

OUT_DIR = ROOT / "reports" / "shap_evaluation"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
SUMMARY = {}   # key numbers for the write-up, saved to summary.json at the end


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")


def save_table(df, name, index=True):
    df.to_csv(OUT_DIR / f"{name}.csv", index=index)


versions = {"python": sys.version.split()[0], "lightgbm": lightgbm.__version__, "shap": shap.__version__,
            "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
            "matplotlib": matplotlib.__version__}
print(versions)

## 1. Load the frozen run and verify it
`load_saved_results` checks the final-selection and completion records and the evidence snapshot hashes. It only reads files: it does not rerun the held-out evaluation. Then the pipeline file and both feature tables are checked against the fingerprints recorded before the test.

In [ ]:
saved = load_saved_results(ROOT)
selection, test_result = saved["selection"], saved["test"]
RUN_DIR = (ROOT / selection["selected_run"]).resolve()
meta = json.loads((RUN_DIR / "selected_metadata.json").read_text())


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


PIPELINE_PATH = RUN_DIR / "selected_pipeline.joblib"
TEST_PATH = (ROOT / "notebooks" / meta["test_path"]).resolve()
TRAIN_PATH = (ROOT / "notebooks" / meta["train_path"]).resolve()
fp = selection["fingerprints_sha256"]
checks = {
    "model / strategy": (meta["model"], meta["strategy"]) == (selection["model"], selection["strategy"]),
    "threshold": meta["threshold"] == selection["threshold"] == test_result["point"]["threshold"],
    "pipeline file": sha256(PIPELINE_PATH) == fp["pipeline"],
    "test feature table": sha256(TEST_PATH) == fp["test"],
    "train feature table": sha256(TRAIN_PATH) == fp["train"],
}
display(pd.Series(checks, name="matches the final selection").to_frame())
assert all(checks.values()), "The run or its inputs differ from the approved final selection. Stop here."

FEATURES = list(meta["feature_order"])
BINARY = list(meta["binary_features"])
RESTORE_NA_COLS = list(meta["restore_na_cols"])
ID_COL, TARGET = meta["id_col"], meta["target"]
THRESHOLD = float(meta["threshold"])
pipeline = joblib.load(PIPELINE_PATH)
print(f"Run: {selection['selected_run']}  |  model: {meta['model']} / {meta['strategy']}  |  threshold: {THRESHOLD:.6f}")
print(pipeline)

## 2. Reproduce the frozen held-out scores
The pipeline is split into its preprocessing steps (the median `SimpleImputer`) and the fitted `LGBMClassifier`. SHAP explains the classifier on the **imputed** matrix, which is exactly what the model sees. Plots still show each provider's **original** value, with missing values kept as missing.

In [ ]:
def load_table(path):
    df = pd.read_csv(path)
    if not pd.api.types.is_numeric_dtype(df[TARGET]):
        df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})
    df[ID_COL] = df[ID_COL].astype(str)
    assert df[ID_COL].is_unique
    return df


def restore_missing(X, cols):
    X = X.copy()
    for col in cols:
        X.loc[X[f"{col}Missing"] == 1, col] = np.nan
    return X


steps = pipeline.steps if hasattr(pipeline, "steps") else [("model", pipeline)]
model = steps[-1][1]
preprocessors = [(name, step) for name, step in steps[:-1]
                 if step is not None and not (isinstance(step, str) and step == "passthrough")
                 and not (hasattr(step, "fit_resample") and not hasattr(step, "transform"))]
print("Preprocessing:", [name for name, _ in preprocessors], "| model:", type(model).__name__)
assert type(model).__module__.startswith("lightgbm"), "Expected a LightGBM model as the final step."


def preprocess(X):
    Xt = X
    for _, step in preprocessors:
        Xt = step.transform(Xt)
    Xt = np.asarray(Xt, dtype=float)
    assert Xt.shape[1] == len(FEATURES), "Preprocessing changed the number of features."
    return pd.DataFrame(Xt, columns=FEATURES, index=X.index)


test = load_table(TEST_PATH)
X_raw = restore_missing(test[FEATURES], RESTORE_NA_COLS)   # what the pipeline receives
X_model = preprocess(X_raw)                                 # what LightGBM sees
y = test[TARGET].to_numpy().astype(int)
ids = test[ID_COL].to_numpy()

score = pipeline.predict_proba(X_raw)[:, 1]
saved_pred = pd.read_csv(RUN_DIR / "test_predictions.csv", dtype={ID_COL: str}).set_index(ID_COL).loc[ids]
max_score_diff = float(np.max(np.abs(score - saved_pred["score"].to_numpy())))
flagged = score >= THRESHOLD
assert max_score_diff <= 1e-9, f"Scores differ from the saved held-out predictions (max diff {max_score_diff:.2e})."
assert (saved_pred["y_true"].to_numpy() == y).all(), "Labels differ from the saved predictions."
assert (saved_pred["flagged"].astype(bool).to_numpy() == flagged).all(), "Flags differ from the saved predictions."

outcome = np.select([flagged & (y == 1), flagged & (y == 0), ~flagged & (y == 1)], ["TP", "FP", "FN"], "TN")
OUTCOMES = ["TP", "FP", "FN", "TN"]
counts = pd.Series(outcome).value_counts().reindex(OUTCOMES).fillna(0).astype(int)
point = test_result["point"]
assert counts.to_dict() == {"TP": point["tp"], "FP": point["fp"], "FN": point["fn"], "TN": point["tn"]}
print(f"Test providers: {len(test)} | fraud labels: {int(y.sum())} | flagged: {int(flagged.sum())}")
print(f"Max |recomputed - saved score| = {max_score_diff:.1e}; confusion counts match test_metrics.json: {counts.to_dict()}")

# How the imputer fills the restored columns (values learned from the 4,328 development providers)
imputer_rows = []
for name, step in preprocessors:
    if hasattr(step, "statistics_"):
        stats = pd.Series(step.statistics_, index=FEATURES)
        for col in RESTORE_NA_COLS:
            imputer_rows.append({"feature": col, "strategy": getattr(step, "strategy", "?"),
                                 "fill_value_from_training": stats[col],
                                 "missing_in_test": int(X_raw[col].isna().sum()),
                                 "missing_share_test": float(X_raw[col].isna().mean())})
imputer_table = pd.DataFrame(imputer_rows).set_index("feature")
display(imputer_table)
SUMMARY["data"] = {"n_test": int(len(test)), "n_fraud": int(y.sum()), "n_flagged": int(flagged.sum()),
                   "counts": counts.to_dict(), "threshold": THRESHOLD,
                   "max_score_diff_vs_saved": max_score_diff, "imputer": imputer_table.reset_index().to_dict("records")}

## 3. Compute SHAP values
`pred_contrib=True` returns one TreeSHAP value per feature plus a final column with the base value: the model's average log-odds over the training data, weighted by tree cover. Two checks follow:
- **Additivity:** the base value plus a provider's contributions should equal its raw log-odds score, and its probability after the sigmoid.
- **Independent implementation:** `shap.TreeExplainer` should give the same values.

In [ ]:
def logit(p):
    return np.log(p / (1 - p))


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


t0 = time.time()
contrib = np.asarray(model.booster_.predict(X_model, pred_contrib=True))
assert contrib.shape == (len(X_model), len(FEATURES) + 1), contrib.shape
sv = pd.DataFrame(contrib[:, :-1], columns=FEATURES, index=X_model.index)   # SHAP values, log-odds
BASE = float(contrib[0, -1])
margin = np.asarray(model.booster_.predict(X_model, raw_score=True))
THRESHOLD_LOGODDS = float(logit(THRESHOLD))

gap_margin = float(np.max(np.abs(sv.sum(axis=1).to_numpy() + BASE - margin)))
gap_prob = float(np.max(np.abs(sigmoid(sv.sum(axis=1).to_numpy() + BASE) - score)))


def as_positive_class(values):
    # shap returns a list (one array per class) or a 3-D array for some versions/models
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    return values[..., -1] if values.ndim == 3 else values


tree_explainer = shap.TreeExplainer(model)
sv_pkg = as_positive_class(tree_explainer.shap_values(X_model))
gap_pkg = float(np.max(np.abs(sv_pkg - sv.to_numpy())))
base_pkg = float(np.atleast_1d(tree_explainer.expected_value)[-1])

print(f"Computed in {time.time() - t0:.1f}s")
print(f"Base value (log-odds): {BASE:.4f}  -> probability {sigmoid(BASE):.4f}  "
      f"(shap package base value: {base_pkg:.4f})")
print(f"Threshold 0.0861 in log-odds: {THRESHOLD_LOGODDS:.4f}")
print(f"Additivity, max gap vs raw log-odds: {gap_margin:.2e}; vs probability: {gap_prob:.2e}")
print(f"pred_contrib vs shap.TreeExplainer, max abs difference: {gap_pkg:.2e}")
assert gap_margin < 1e-6 and gap_prob < 1e-9

SUMMARY["shap"] = {"base_value_logodds": BASE, "base_value_prob": float(sigmoid(BASE)),
                   "threshold_logodds": THRESHOLD_LOGODDS, "additivity_gap_logodds": gap_margin,
                   "additivity_gap_prob": gap_prob, "shap_package_max_diff": gap_pkg,
                   "shap_package_base_value": base_pkg}
explanation = shap.Explanation(values=sv.to_numpy(), base_values=np.full(len(sv), BASE),
                               data=X_raw.to_numpy(dtype=float), feature_names=FEATURES)
per_provider = pd.DataFrame({ID_COL: ids, "y_true": y, "score": score, "logodds": margin,
                             "flagged": flagged, "outcome": outcome})
save_table(pd.concat([per_provider, sv.add_prefix("shap_")], axis=1), "shap_values_test", index=False)

## 4. Global importance
Global importance is the mean absolute SHAP value: the average size of a feature's push on the log-odds score, in either direction. It is reported with:
- a 95% interval and a rank interval from 1,000 label-stratified provider bootstraps (fixed model and SHAP values, so this covers sampling of the test providers only);
- the mean **signed** SHAP value among fraud-labelled and non-fraud providers;
- the **direction** of the effect: the Spearman correlation between a feature's value and its SHAP value (non-missing values only). A positive value means higher values of the feature raise the risk score;
- LightGBM's built-in **gain** and **split** importances, for comparison.

In [ ]:
A = sv.abs().to_numpy()
rng = np.random.default_rng(SEED)
pos_idx, neg_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
boot_mean = np.empty((N_BOOT, len(FEATURES)))
boot_rank = np.empty((N_BOOT, len(FEATURES)))
for b in range(N_BOOT):
    idx = np.concatenate([rng.choice(pos_idx, len(pos_idx)), rng.choice(neg_idx, len(neg_idx))])
    m = A[idx].mean(axis=0)
    boot_mean[b] = m
    boot_rank[b] = (-m).argsort().argsort() + 1


def direction(f):
    x = X_raw[f]
    ok = x.notna()
    if x[ok].nunique() < 2:
        return np.nan
    return spearmanr(x[ok], sv.loc[ok, f]).statistic


gain = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=FEATURES)
split = pd.Series(model.booster_.feature_importance(importance_type="split"), index=FEATURES)
mean_abs = sv.abs().mean()
glob = pd.DataFrame({
    "mean_abs_shap": mean_abs,
    "ci_low": np.percentile(boot_mean, 2.5, axis=0),
    "ci_high": np.percentile(boot_mean, 97.5, axis=0),
    "share_pct": 100 * mean_abs / mean_abs.sum(),
    "rank": mean_abs.rank(ascending=False, method="first").astype(int),
    "rank_ci_low": np.percentile(boot_rank, 2.5, axis=0).astype(int),
    "rank_ci_high": np.percentile(boot_rank, 97.5, axis=0).astype(int),
    "mean_shap_fraud": sv[y == 1].mean(),
    "mean_shap_nonfraud": sv[y == 0].mean(),
    "direction_spearman": [direction(f) for f in FEATURES],
    "gain_share_pct": 100 * gain / gain.sum(),
    "gain_rank": gain.rank(ascending=False, method="first").astype(int),
    "split_count": split,
}).sort_values("rank").rename_axis("feature")
save_table(glob, "global_importance")
display(glob)

rho_gain = spearmanr(glob["rank"], glob["gain_rank"]).statistic
top_shap, top_gain = set(glob.index[:TOP_K]), set(glob.sort_values("gain_rank").index[:TOP_K])
top3_share = float(glob["share_pct"].iloc[:3].sum())
print(f"Top 3 features carry {top3_share:.1f}% of total mean |SHAP|; top {TOP_K} carry "
      f"{glob['share_pct'].iloc[:TOP_K].sum():.1f}%.")
print(f"Spearman rank correlation, SHAP vs gain importance: {rho_gain:.3f}; "
      f"top-{TOP_K} overlap: {len(top_shap & top_gain)}/{TOP_K}")
unused = glob.index[glob["split_count"] == 0].tolist()
print("Features never used in a split:", unused or "none")
SUMMARY["global"] = {"table": glob.reset_index(names="feature").to_dict("records"),
                     "top3_share_pct": top3_share, "spearman_shap_vs_gain": rho_gain,
                     "topk_overlap_shap_gain": len(top_shap & top_gain), "unused_features": unused}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), gridspec_kw={"width_ratios": [1, 1]})
g = glob.iloc[::-1]
axes[0].barh(g.index, g["mean_abs_shap"], xerr=[g["mean_abs_shap"] - g["ci_low"], g["ci_high"] - g["mean_abs_shap"]],
             color="#3b6ea5", ecolor="#555", capsize=2)
axes[0].set_xlabel("mean |SHAP| (log-odds)")
axes[0].set_title("SHAP importance, held-out test (95% bootstrap CI)")
gg = glob.sort_values("gain_rank").iloc[::-1]
axes[1].barh(gg.index, gg["gain_share_pct"], color="#9aa5b1")
axes[1].set_xlabel("share of total split gain (%)")
axes[1].set_title("LightGBM gain importance (training)")
fig.tight_layout()
save_fig(fig, "01_global_importance")
plt.show()

shap.plots.beeswarm(explanation, max_display=15, show=False)
fig = plt.gcf()
fig.suptitle("SHAP values per provider (colour = feature value; grey = missing)", y=1.0)
save_fig(fig, "02_beeswarm")
plt.show()

## 5. Correlated features and feature groups
SHAP divides credit between correlated features in a way that depends on how the trees happened to split. For example, four of the claim-amount features summarise the same distribution. Credit for a shared signal can therefore move between them. Summing SHAP values within a domain group is still exact, because contributions are additive. This makes a **group** ranking more stable to interpret than individual ranks inside a correlated block.

`cancellation` compares the sum of members' mean |SHAP| with the group's mean |SHAP|. A ratio well above 1 means members often push in opposite directions for the same provider.

In [ ]:
GROUPS = {
    "Volume & patient concentration": ["NumClaims", "AvgClaimsPerBeneficiary", "TopBeneficiaryClaimShare"],
    "Claim amounts": ["AvgClaimAmt", "MedianClaimAmt", "StdClaimAmt", "MaxClaimAmt", "DeductibleToReimbursementRatio"],
    "Care setting & stay length": ["PropInpatient", "AvgClaimDurationDays", "AvgLengthOfStayDays"],
    "Coding patterns": ["AvgNumDiagnosisCodes", "AvgNumProcedureCodes", "TopDiagnosisCodeShare",
                        "TopProcedureCodeShare", "TopDiagnosisCodeShareMissing", "TopProcedureCodeShareMissing"],
    "Physician network": ["AvgNumPhysiciansInvolved", "NumUniqueAttendingPhysicians",
                          "NumUniqueOperatingPhysicians", "AvgPhysicianProviderSpread"],
    "Beneficiary profile": ["AvgAgeOfBeneficiaries", "PropDeceasedBeneficiaries", "AvgChronicConditions"],
    "Benford's Law": ["BenfordMAD", "BenfordMADMissing"],
}
members = [f for cols in GROUPS.values() for f in cols]
assert sorted(members) == sorted(FEATURES) and len(members) == len(set(members)), "Groups must partition the features."
group_of = {f: g for g, cols in GROUPS.items() for f in cols}

group_sv = pd.DataFrame({g: sv[cols].sum(axis=1) for g, cols in GROUPS.items()})
group_tbl = pd.DataFrame({
    "n_features": {g: len(c) for g, c in GROUPS.items()},
    "mean_abs_shap": group_sv.abs().mean(),
    "sum_member_mean_abs": {g: mean_abs[c].sum() for g, c in GROUPS.items()},
    "mean_shap_fraud": group_sv[y == 1].mean(),
    "mean_shap_nonfraud": group_sv[y == 0].mean(),
    "top_member": {g: mean_abs[c].idxmax() for g, c in GROUPS.items()},
})
group_tbl["share_pct"] = 100 * group_tbl["mean_abs_shap"] / group_tbl["mean_abs_shap"].sum()
group_tbl["cancellation"] = group_tbl["sum_member_mean_abs"] / group_tbl["mean_abs_shap"]
group_tbl = group_tbl.sort_values("mean_abs_shap", ascending=False).rename_axis("group")
save_table(group_tbl, "group_importance")
display(group_tbl)

corr = X_raw.corr(method="spearman")
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
         .rename("spearman").reset_index().rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
pairs = pairs[pairs["spearman"].abs() >= 0.7].sort_values("spearman", key=np.abs, ascending=False)
pairs["shap_rank_a"] = pairs["feature_a"].map(glob["rank"])
pairs["shap_rank_b"] = pairs["feature_b"].map(glob["rank"])
save_table(pairs, "correlated_pairs", index=False)
print("Feature pairs with |Spearman| >= 0.7 on the test providers:")
display(pairs)

fig, ax = plt.subplots(figsize=(8, 4.5))
gt = group_tbl.iloc[::-1]
ax.barh(gt.index, gt["sum_member_mean_abs"], color="#d5dbe1", label="sum of members' mean |SHAP|")
ax.barh(gt.index, gt["mean_abs_shap"], color="#3b6ea5", label="group mean |SHAP|")
ax.set_xlabel("log-odds")
ax.set_title("Importance by feature group")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()
save_fig(fig, "03_group_importance")
plt.show()
SUMMARY["groups"] = {"table": group_tbl.reset_index(names="group").to_dict("records"),
                     "correlated_pairs": pairs.to_dict("records")}

## 6. Dependence plots and domain features
A dependence plot shows how a feature's SHAP value changes with its value. Each point is one provider and the black line is the binned mean. Point colour shows the **label**, which the model never sees. It shows whether the providers the model pushes up are in fact the labelled ones. Missing values are not plotted; the panel title reports their count and mean SHAP.

The section ends with three checks:
- the three **missing-value columns**, where the value and its flag act together;
- the two features the project introduced as less standard signals: `BenfordMAD`, Benford's Law deviation of claim amounts, and `AvgPhysicianProviderSpread`, a physician–provider network feature;
- SHAP **interaction values**, to find the feature pairs whose joint effect matters most.

In [ ]:
LABEL_COLOURS = {0: "#9aa5b1", 1: "#c0392b"}


def dependence_panel(ax, f):
    x, s = X_raw[f], sv[f]
    ok = x.notna()
    xv, sv_f, yv = x[ok].to_numpy(dtype=float), s[ok].to_numpy(), y[ok.to_numpy()]
    if f in BINARY or x[ok].nunique() <= 2:
        jitter = np.random.default_rng(SEED).uniform(-0.12, 0.12, len(xv))
        for lab in (0, 1):
            m = yv == lab
            ax.scatter(xv[m] + jitter[m], sv_f[m], s=7, alpha=0.5, color=LABEL_COLOURS[lab], lw=0)
        ax.set_xticks(sorted(np.unique(xv)))
    else:
        log = xv.min() > 0 and np.nanpercentile(xv, 99) / max(np.nanmedian(xv), 1e-12) > 20
        for lab in (0, 1):
            m = yv == lab
            ax.scatter(xv[m], sv_f[m], s=7, alpha=0.5 if lab == 0 else 0.8, color=LABEL_COLOURS[lab], lw=0)
        bins = pd.qcut(pd.Series(xv).rank(method="first"), q=min(12, max(2, len(xv) // 40)), labels=False)
        line = pd.DataFrame({"x": xv, "s": sv_f, "b": bins}).groupby("b").mean().sort_values("x")
        ax.plot(line["x"], line["s"], color="black", lw=1.4)
        if log:
            ax.set_xscale("log")
    ax.axhline(0, color="#555", lw=0.6, ls=":")
    n_miss = int((~ok).sum())
    title = f"{f}\n(rank {glob.loc[f, 'rank']}; mean |SHAP| {mean_abs[f]:.3f})"
    if n_miss:
        title += f"\n{n_miss} missing, mean SHAP {s[~ok].mean():+.3f}"
    ax.set_title(title, fontsize=9)
    ax.set_ylabel("SHAP (log-odds)", fontsize=8)
    ax.tick_params(labelsize=8)


top_features = glob.index[:N_DEPENDENCE].tolist()
ncols = 3
nrows = int(np.ceil(len(top_features) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.2 * nrows))
for ax, f in zip(np.ravel(axes), top_features):
    dependence_panel(ax, f)
for ax in np.ravel(axes)[len(top_features):]:
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=l)
           for l, c in [("not fraud-labelled", LABEL_COLOURS[0]), ("fraud-labelled", LABEL_COLOURS[1])]]
fig.legend(handles=handles, loc="upper right", frameon=False)
fig.suptitle("Dependence plots, top features by mean |SHAP|", y=1.0)
fig.tight_layout()
save_fig(fig, "04_dependence_top")
plt.show()

# Threshold-like behaviour: where along each top feature does the binned mean SHAP cross zero?
cross_rows = []
for f in top_features:
    ok = X_raw[f].notna()
    if f in BINARY or X_raw.loc[ok, f].nunique() <= 2:
        continue
    d = pd.DataFrame({"x": X_raw.loc[ok, f], "s": sv.loc[ok, f]})
    d["bin"] = pd.qcut(d["x"].rank(method="first"), 10, labels=False)
    b = d.groupby("bin").agg(x_min=("x", "min"), x_max=("x", "max"), mean_shap=("s", "mean"))
    b.insert(0, "feature", f)
    cross_rows.append(b.reset_index())
deciles = pd.concat(cross_rows, ignore_index=True) if cross_rows else pd.DataFrame()
save_table(deciles, "dependence_deciles", index=False)
SUMMARY["dependence_deciles"] = deciles.to_dict("records")

In [ ]:
# Missing-value columns: the value (imputed to the training median when missing) and its flag act together
miss_rows = []
for col in RESTORE_NA_COLS:
    flag = f"{col}Missing"
    miss = X_raw[col].isna().to_numpy()
    for status, m in (("missing", miss), ("present", ~miss)):
        if not m.any():
            continue
        miss_rows.append({
            "feature": col, "status": status, "n": int(m.sum()),
            "fraud_rate": float(y[m].mean()),
            "mean_shap_value": float(sv.loc[m, col].mean()),
            "mean_shap_flag": float(sv.loc[m, flag].mean()),
            "mean_shap_combined": float((sv.loc[m, col] + sv.loc[m, flag]).mean()),
            "mean_abs_combined": float((sv.loc[m, col] + sv.loc[m, flag]).abs().mean()),
        })
missing_tbl = pd.DataFrame(miss_rows).set_index(["feature", "status"])
save_table(missing_tbl, "missing_value_effects")
display(missing_tbl)

# Domain features introduced by the project
DOMAIN = ["BenfordMAD", "AvgPhysicianProviderSpread"]
domain_tbl = glob.loc[DOMAIN + ["BenfordMADMissing"], ["rank", "rank_ci_low", "rank_ci_high", "mean_abs_shap",
                                                       "share_pct", "direction_spearman", "gain_rank"]]
display(domain_tbl)
fig, axes = plt.subplots(1, len(DOMAIN), figsize=(10, 4))
for ax, f in zip(np.atleast_1d(axes), DOMAIN):
    dependence_panel(ax, f)
fig.suptitle("Domain features: Benford deviation and physician–provider spread", y=1.02)
fig.tight_layout()
save_fig(fig, "05_dependence_domain")
plt.show()
SUMMARY["missing"] = missing_tbl.reset_index().to_dict("records")
SUMMARY["domain"] = domain_tbl.reset_index(names="feature").to_dict("records")

In [ ]:
# SHAP interaction values (path-dependent). Off-diagonal entries are split symmetrically, so a pair's
# total interaction is twice one entry.
interaction_top = None
if RUN_INTERACTIONS:
    t0 = time.time()
    try:
        raw_inter = tree_explainer.shap_interaction_values(X_model)
        inter = np.asarray(raw_inter[-1] if isinstance(raw_inter, list) else raw_inter)
        if inter.ndim == 4:
            inter = inter[..., -1] if inter.shape[-1] == 2 else inter[-1]
        assert inter.shape == (len(X_model), len(FEATURES), len(FEATURES)), inter.shape
        mean_abs_inter = np.abs(inter).mean(axis=0)
        main_effect = pd.Series(np.diag(mean_abs_inter), index=FEATURES)
        iu = np.triu_indices(len(FEATURES), k=1)
        interaction_top = (pd.DataFrame({"feature_a": np.array(FEATURES)[iu[0]],
                                         "feature_b": np.array(FEATURES)[iu[1]],
                                         "mean_abs_interaction": 2 * mean_abs_inter[iu]})
                           .sort_values("mean_abs_interaction", ascending=False).head(TOP_K).reset_index(drop=True))
        total_inter = float(2 * mean_abs_inter[iu].sum())
        print(f"Interaction values computed in {time.time() - t0:.0f}s")
        print(f"Sum of mean |main effects|: {main_effect.sum():.3f}; sum of mean |pairwise interactions|: {total_inter:.3f}")
        display(interaction_top)
        save_table(interaction_top, "top_interactions", index=False)
        SUMMARY["interactions"] = {"top": interaction_top.to_dict("records"),
                                   "sum_main": float(main_effect.sum()), "sum_pairwise": total_inter}
    except Exception as exc:  # optional analysis; record why it was skipped
        print(f"Interaction values not computed: {type(exc).__name__}: {exc}")
        SUMMARY["interactions"] = {"error": f"{type(exc).__name__}: {exc}"}

## 7. Errors and case studies
The labels now group the explanations into four outcomes at the frozen threshold: true positives (TP), false positives (FP), false negatives (FN) and true negatives (TN). The questions are:
- **False positives:** what makes them look like fraud, and do they resemble the true positives?
- **False negatives:** what keeps them below the threshold?
- **Fragile decisions:** how many decisions rest on one feature? A decision is counted as single-feature when one contribution, in the direction of the decision, is larger than the provider's distance from the threshold in log-odds. Removing that contribution from the additive sum would flip the flag. This is a statement about the attribution, not a counterfactual: changing the feature's value would change other contributions too.

In [ ]:
top12 = glob.index[:12].tolist()
profile = pd.DataFrame({o: sv.loc[outcome == o, top12].mean() for o in OUTCOMES})
medians = pd.DataFrame({o: X_raw.loc[outcome == o, top12].median() for o in OUTCOMES})
profile_out = profile.add_prefix("mean_shap_").join(medians.add_prefix("median_value_")).rename_axis("feature")
save_table(profile_out, "outcome_profile")
display(profile_out)

fig, ax = plt.subplots(figsize=(7.5, 6))
lim = float(np.abs(profile.to_numpy()).max())
im = ax.imshow(profile.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(OUTCOMES)), [f"{o}\n(n={counts[o]})" for o in OUTCOMES])
ax.set_yticks(range(len(top12)), top12)
for i in range(len(top12)):
    for j in range(len(OUTCOMES)):
        v = profile.iat[i, j]
        ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.6 * lim else "black")
fig.colorbar(im, ax=ax, label="mean SHAP (log-odds)")
ax.set_title("Mean SHAP by outcome at the frozen threshold")
fig.tight_layout()
save_fig(fig, "06_outcome_profile")
plt.show()

# Group-level view by outcome (sums within domain groups are exact)
group_profile = pd.DataFrame({o: group_sv[outcome == o].mean() for o in OUTCOMES})
group_profile.loc["(base value)"] = BASE
group_profile.loc["(mean log-odds score)"] = [margin[outcome == o].mean() for o in OUTCOMES]
save_table(group_profile, "outcome_group_profile")
display(group_profile)

# Most frequent top positive driver among flagged providers (what the dashboard lists first)
top_driver = sv.idxmax(axis=1)
driver_tbl = (pd.crosstab(top_driver[flagged], outcome[flagged]).reindex(columns=["TP", "FP"], fill_value=0)
              .assign(total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False)
              .rename_axis(index="top_positive_driver", columns=None))
save_table(driver_tbl, "flagged_top_driver")
display(driver_tbl.head(TOP_K))
SUMMARY["outcomes"] = {"profile": profile_out.reset_index(names="feature").to_dict("records"),
                       "group_profile": group_profile.reset_index(names="group").to_dict("records"),
                       "flagged_top_driver": driver_tbl.reset_index(names="feature").to_dict("records")}

In [ ]:
# Fragility: does one contribution decide the flag?
distance = margin - THRESHOLD_LOGODDS            # >= 0 when flagged
toward = np.where(flagged[:, None], sv.to_numpy(), -sv.to_numpy())   # contributions in the decision's direction
largest_toward = toward.max(axis=1)
single_feature = largest_toward > np.abs(distance)
near = np.abs(distance) <= NEAR_BAND_LOGODDS

frag = pd.DataFrame({
    "n": counts,
    "near_threshold": pd.Series(near).groupby(outcome).sum(),
    "single_feature_decision": pd.Series(single_feature).groupby(outcome).sum(),
    "median_distance_logodds": pd.Series(np.abs(distance)).groupby(outcome).median(),
}).reindex(OUTCOMES).rename_axis("outcome")
frag["single_feature_pct"] = 100 * frag["single_feature_decision"] / frag["n"]
save_table(frag, "decision_fragility")
display(frag)
deciding = pd.Series(np.array(FEATURES)[toward.argmax(axis=1)][single_feature]).value_counts().head(TOP_K)
print("Feature that alone decides the flag, most frequent:")
display(deciding.to_frame("providers"))

# Importance among providers near the threshold (where review decisions are actually made)
near_imp = sv[near].abs().mean().sort_values(ascending=False)
near_tbl = pd.DataFrame({"mean_abs_shap_near": near_imp, "rank_near": np.arange(1, len(near_imp) + 1),
                         "rank_all": glob.loc[near_imp.index, "rank"]}).rename_axis("feature")
print(f"{int(near.sum())} providers within ±{NEAR_BAND_LOGODDS} log-odds of the threshold "
      f"(scores {sigmoid(THRESHOLD_LOGODDS - NEAR_BAND_LOGODDS):.3f}–{sigmoid(THRESHOLD_LOGODDS + NEAR_BAND_LOGODDS):.3f}).")
display(near_tbl.head(TOP_K))
save_table(near_tbl, "importance_near_threshold")
SUMMARY["fragility"] = {"table": frag.reset_index(names="outcome").to_dict("records"),
                        "deciding_features": deciding.to_dict(), "n_near": int(near.sum()),
                        "near_band_scores": [float(sigmoid(THRESHOLD_LOGODDS - NEAR_BAND_LOGODDS)),
                                             float(sigmoid(THRESHOLD_LOGODDS + NEAR_BAND_LOGODDS))],
                        "near_importance": near_tbl.head(TOP_K).reset_index(names="feature").to_dict("records")}

### Case studies
The five providers below are chosen by fixed rules, not by hand, so the selection is reproducible:
1. the highest-scoring true positive;
2. the true positive closest above the threshold;
3. the highest-scoring false positive;
4. the false negative closest below the threshold (a near miss);
5. the lowest-scoring false negative (a confident miss).

Each waterfall starts at the base value and adds the largest contributions. The remaining features are pooled into one bar. Feature values are shown as the provider's original values; `nan` means the value was missing and the model used the training median.

In [ ]:
def pick(o, by, largest=True):
    idx = np.flatnonzero(outcome == o)
    if len(idx) == 0:
        return None
    vals = by[idx]
    return int(idx[np.argmax(vals) if largest else np.argmin(vals)])


CASES = {
    "Highest-scoring true positive": pick("TP", score),
    "True positive closest above the threshold": pick("TP", score, largest=False),
    "Highest-scoring false positive": pick("FP", score),
    "False negative closest below the threshold": pick("FN", score),
    "Lowest-scoring false negative": pick("FN", score, largest=False),
}
case_rows = []
for k, (label, i) in enumerate(CASES.items(), start=1):
    if i is None:
        continue
    row = sv.iloc[i]
    top = row.abs().sort_values(ascending=False).index[:5]
    case_rows.append({"case": label, ID_COL: ids[i], "outcome": outcome[i], "score": score[i],
                      "logodds_minus_threshold": distance[i],
                      "top_drivers": "; ".join(f"{f}={X_raw.iloc[i][f]:.4g} ({row[f]:+.2f})" for f in top)})
    shap.plots.waterfall(explanation[i], max_display=10, show=False)
    fig = plt.gcf()
    fig.suptitle(f"{k}. {label}: {ids[i]} — score {score[i]:.3f} ({outcome[i]})", y=1.02, fontsize=11)
    save_fig(fig, f"07_case_{k}_{outcome[i]}")
    plt.show()
cases = pd.DataFrame(case_rows)
save_table(cases, "case_studies", index=False)
with pd.option_context("display.max_colwidth", None):
    display(cases)
SUMMARY["cases"] = cases.to_dict("records")

## 8. Robustness checks
1. **Additivity and implementation:** shown in section 3.
2. **Ranking uncertainty:** the bootstrap rank intervals in section 4.
3. **Path-dependent vs interventional SHAP.** Path-dependent TreeSHAP takes the expectation over the training distribution as stored in the trees. With correlated features, it can give credit to a feature the model does not use directly. Interventional TreeSHAP instead breaks feature dependence using a background sample of development providers. It answers a more causal-in-the-model question: what happens to the score if this feature is replaced by background values? If both methods agree on the ranking, the conclusions do not hinge on that choice. The background is a random sample of the development providers. Only their features are used, never their labels.

In [ ]:
train = load_table(TRAIN_PATH)
background = preprocess(restore_missing(train[FEATURES], RESTORE_NA_COLS)).sample(BACKGROUND_SIZE, random_state=SEED)
t0 = time.time()
int_explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional", model_output="raw")
sv_int = pd.DataFrame(as_positive_class(int_explainer.shap_values(X_model, check_additivity=False)),
                      columns=FEATURES, index=X_model.index)
base_int = float(np.atleast_1d(int_explainer.expected_value)[-1])
gap_int = float(np.max(np.abs(sv_int.sum(axis=1).to_numpy() + base_int - margin)))
print(f"Interventional SHAP computed in {time.time() - t0:.0f}s; base value {base_int:.4f}; additivity gap {gap_int:.2e}")

mean_abs_int = sv_int.abs().mean()
cmp = pd.DataFrame({"path_dependent": mean_abs, "interventional": mean_abs_int})
cmp["rank_path"] = cmp["path_dependent"].rank(ascending=False, method="first").astype(int)
cmp["rank_interventional"] = cmp["interventional"].rank(ascending=False, method="first").astype(int)
cmp = cmp.sort_values("rank_path").rename_axis("feature")
rho_methods = spearmanr(cmp["path_dependent"], cmp["interventional"]).statistic
overlap = len(set(cmp.index[:TOP_K]) & set(cmp.sort_values("rank_interventional").index[:TOP_K]))
same_top1 = float((sv.abs().idxmax(axis=1) == sv_int.abs().idxmax(axis=1)).mean())
row_corr = np.array([spearmanr(a, b).statistic for a, b in zip(sv.to_numpy(), sv_int.to_numpy())])
save_table(cmp, "path_vs_interventional")
display(cmp)
print(f"Spearman correlation of global importance (path vs interventional): {rho_methods:.3f}; "
      f"top-{TOP_K} overlap {overlap}/{TOP_K}")
print(f"Same top driver for {same_top1:.1%} of providers; median per-provider Spearman of SHAP vectors {np.median(row_corr):.3f}")

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(cmp["path_dependent"], cmp["interventional"], color="#3b6ea5")
for f, r in cmp.head(TOP_K).iterrows():
    ax.annotate(f, (r["path_dependent"], r["interventional"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
lim = float(cmp[["path_dependent", "interventional"]].to_numpy().max()) * 1.08
ax.plot([0, lim], [0, lim], color="#999", ls="--", lw=0.8)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("mean |SHAP|, path-dependent (log-odds)")
ax.set_ylabel(f"mean |SHAP|, interventional, {BACKGROUND_SIZE} background providers")
ax.set_title("Global importance under two SHAP variants")
fig.tight_layout()
save_fig(fig, "08_path_vs_interventional")
plt.show()
SUMMARY["robustness"] = {"background_size": BACKGROUND_SIZE, "interventional_base": base_int,
                         "interventional_additivity_gap": gap_int, "spearman_global": rho_methods,
                         "topk_overlap": overlap, "same_top_driver_share": same_top1,
                         "median_row_spearman": float(np.median(row_corr)),
                         "table": cmp.reset_index(names="feature").to_dict("records")}

## 9. Save the summary
`summary.json` collects the numbers quoted in the write-up below and in `reports/shap_evaluation.md`.

In [ ]:
SUMMARY["run"] = {"selected_run": selection["selected_run"], "versions": versions, "seed": SEED, "n_boot": N_BOOT,
                  "near_band_logodds": NEAR_BAND_LOGODDS, "generated_at": pd.Timestamp.now(tz="Asia/Singapore").isoformat()}


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o


(OUT_DIR / "summary.json").write_text(json.dumps(_clean(SUMMARY), indent=2))
print("Saved:", sorted(p.relative_to(ROOT).as_posix() for p in OUT_DIR.rglob("*") if p.is_file()))

## 10. Write-up
*Added after the notebook has been executed; see also `reports/shap_evaluation.md`.*